# 🧪 IV-LLM Pipeline: Automated Instrumental Variable Discovery

This notebook demonstrates how to use the **IV-LLM pipeline** in the `causal-agent` framework. The IV-LLM pipeline is a multi-agent framework designed to automate the discovery and validation of **Instrumental Variables (IVs)** using Large Language Models.

## 🛠️ The IV-LLM Workflow

When the `--iv_llm` flag is enabled, the agent follows a specialized sub-pipeline:

1.  **Hypothesis Generation**: The LLM hypothesizes potential instruments based on dataset context and variable names. It looks for variables that might be correlated with the treatment but not directly with the outcome.
2.  **Confounder Mining**: It proactively identifies potential confounders that might violate the fundamental IV assumptions (Independence or Exclusion restrictions).
3.  **Critic Validation**: Specialized LLM "critics" analyze each candidate:
    *   **Independence Critic**: Reasons about whether the instrument is truly independent of unobserved confounders.
    *   **Exclusion Critic**: Reasons about whether the instrument's effect on the outcome happens *only* through the treatment.
4.  **Final Selection**: The agent selects the most robust instrument candidates for the final estimation stage.

---

## 1. Setup

First, we'll ensure the environment is set up. Make sure you have your API keys in a `.env` file.

In [ ]:
import os
import pandas as pd
from dotenv import load_dotenv
from cais.agent import CausalAgent

# Load environment variables (API keys)
load_dotenv()

# Check if API keys are available
if not os.getenv("OPENAI_API_KEY") and not os.getenv("ANTHROPIC_API_KEY"):
    print("⚠️ Warning: No LLM API keys found. Please set them in your .env file.")

## 2. Example Dataset: Compulsory Schooling (AK91)

We will use the **AK91** dataset (from Angrist & Krueger, 1991), a classic example in causal inference literature. 

**The Causal Question**: How much more does a person earn for each extra year of education?

**The Challenge**: "Education" is endogenous. People who choose more education might have higher innate "ability," which also leads to higher wages. Measuring the simple correlation would overstate the effect of education.

**The IV Approach**: We use **Quarter of Birth** as an instrument. Because of school entry laws, children born late in the year start school older and turn 16 (legal dropout age) after completing *more* schooling than those born early in the year.

In [ ]:
dataset_path = "../../data/all_data/ak91.csv"
df = pd.read_csv(dataset_path)

print(f"Dataset shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

dataset_description = """
The dataset contains information on individuals' log wages, years of schooling, year of birth, quarter of birth, and state of birth. 
The purpose of the analysis is to estimate the effect of education on wage by taking advantage of US compulsory attendance law. 
Usually, kids born at the beginning of the year (Q1) will enter school at an older age but can drop out as soon as they turn 16. 
The result is that people born later in the year have, on average, more years of education than those born in the beginning of the year.
"""

query = "How much more does a person earn for each extra year of education?"

## 3. Running the Causal Agent with IV-LLM enabled

We initialize the `CausalAgent` and set `use_iv_pipeline=True`. This tells the agent to use the multi-agent discovery process rather than just relying on simple heuristics or direct variable mentions.

In [ ]:
# Initialize the Agent
agent = CausalAgent(
    dataset_path=dataset_path,
    dataset_description=dataset_description,
    use_iv_pipeline=True,  # <--- This enables the IV-LLM pipeline
    model_name="gpt-4o-mini", # Or your preferred model
    provider="openai"
)

# Run the full analysis
output = agent.run_analysis(query=query)

# Display the final explanation
print("\n--- Final Interpretation ---\n")
print(output["explanation"])

## 4. Inspecting the Discovery Results

We can inspect which variables the agent identified as the treatment, outcome, and importantly, the **instrumental variable** discovered via the LLM pipeline.

In [ ]:
print(f"Selected Method: {agent.selected_method}")
print(f"Treatment: {agent.variables.treatment_variable}")
print(f"Outcome: {agent.variables.outcome_variable}")
print(f"Discovered Instrument: {agent.variables.instrument_variable}")

## 📚 Citation

If you use this pipeline, please cite the original paper:

> **IV Co-Scientist: Multi-Agent LLM Framework for Causal Instrumental Variable Discovery**  
> Sheth, Ivaxi and Jin, Zhijing and Wilder, Bryan and Janzing, Dominik and Fritz, Mario  
> *arXiv preprint arXiv:2602.07943*, 2026.